# Mini FSD training on Colab

Thin by design (ADR 0007): this notebook mounts Drive, installs the repo at a
pinned commit, unpacks the data and calls a `minifsd.train` module. All the
logic is in the repo, tested locally.

Before the first run, upload the packs from `data/packs/` (made by
`python -m minifsd.train.pack`) to `My Drive/mini-fsd/packs/`.

Runtime > Change runtime type > T4 GPU. If the session drops, run all cells
again: training resumes from the last checkpoint on Drive.

In [5]:
# Settings for this run. COMMIT pins the code, so a result can be traced
# back to it; use the commit you pushed last (git rev-parse --short HEAD).
COMMIT = "main"
PACKS = ["v9.zip", "coco-people.zip", "v10.zip"]
DRIVE = "/content/drive/MyDrive/mini-fsd"
RUN = "e2a-segformer-b0"  # checkpoints and logs go to DRIVE/runs/RUN

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [7]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# The repo at the pinned commit. Extra packages, pinned, are added here per
# experiment and recorded in ADR 0007.
# --force-reinstall: a new COMMIT must replace the old install even though
# the version number is the same. --no-deps: keep Colab's numpy and scipy.
!pip install -q --force-reinstall --no-deps "git+https://github.com/SegunAdewola/mini-fsd@{COMMIT}"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
import pathlib
import shutil

from minifsd.train import pack

data = pathlib.Path("/content/data")
data.mkdir(exist_ok=True)
for name in PACKS:
    local = data / name
    if not local.exists():
        shutil.copy(f"{DRIVE}/packs/{name}", local)
    folder = data / name.removesuffix(".zip")
    if not folder.exists():
        print("unpacked", pack.unpack(local, data))

## E2a: SegFormer-B0 floor segmentation

Smoke run first: 64 frames, one epoch, on the session disk. It checks the
whole loop on the GPU in a minute or two before the real run.

In [10]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir /content/smoke --epochs 1 --limit 64

config.json: 100% 70.0k/70.0k [00:00<00:00, 78.5MB/s]
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.

pytorch_model.bin: downloading bytes:  86% 12.4M/14.4M [00:00<00:00, 19.7MB/s]
pytorch_model.bin: downloading bytes: 100% 13.5M/13.5M [00:00<00:00, 19.0MB/s, 1.32MB/s  ]
pytorch_model.bin: reconstructing file: 100% 14.4M/14.4M [00:00<00:00, 20.2MB/s, 1.41MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 23814.36it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_he

The real run: all 9,000 v9 train frames, 20 epochs, checkpoints on Drive.
If the session drops, run every cell again from the top: it resumes after
the last finished epoch.

In [11]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir {DRIVE}/runs/{RUN} --epochs 20

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100% 192/192 [00:00<00:00, 23804.50it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.classifier.bias            

In [12]:
# One line per finished epoch; best/ holds the highest val IoU so far.
!cat {DRIVE}/runs/{RUN}/log.jsonl

{"epoch": 0, "train_loss": 0.05215786155561606, "val_iou": 0.9768741412862171, "best_iou": 0.9768741412862171, "seconds": 82.0, "device": "Tesla T4"}
{"epoch": 1, "train_loss": 0.02140828567246596, "val_iou": 0.9840429411680561, "best_iou": 0.9840429411680561, "seconds": 73.3, "device": "Tesla T4"}
{"epoch": 2, "train_loss": 0.015279178099499809, "val_iou": 0.9852410425410074, "best_iou": 0.9852410425410074, "seconds": 74.3, "device": "Tesla T4"}
{"epoch": 3, "train_loss": 0.014534363969332642, "val_iou": 0.9873176543044618, "best_iou": 0.9873176543044618, "seconds": 74.1, "device": "Tesla T4"}
{"epoch": 4, "train_loss": 0.011765132310075893, "val_iou": 0.9881484839473546, "best_iou": 0.9881484839473546, "seconds": 74.9, "device": "Tesla T4"}
{"epoch": 5, "train_loss": 0.010633069614155425, "val_iou": 0.9904128094578222, "best_iou": 0.9904128094578222, "seconds": 74.9, "device": "Tesla T4"}
{"epoch": 6, "train_loss": 0.01096650639259153, "val_iou": 0.9886441483214091, "best_iou": 0.990

## E2b: frozen DINOv2 ViT-S/14 with a floor head

Same data and loop as E2a; only the head trains. Smoke run first, then the
real run (10 epochs). Run the settings, Drive, install and unpack cells
first if this is a new session.

In [16]:
!python -m minifsd.train.dino --train /content/data/v9/train --val /content/data/v9/val --run-dir /content/smoke-dino --epochs 1 --limit 64

config.json: 100% 547/547 [00:00<00:00, 2.36MB/s]

model.safetensors: downloading bytes:  84% 74.0M/88.2M [00:00<00:00, 128MB/s, 4.74MB/s  ] 
model.safetensors: downloading bytes: 100% 83.7M/83.7M [00:00<00:00, 91.9MB/s, 8.10MB/s  ]
model.safetensors: reconstructing file: 100% 88.2M/88.2M [00:00<00:00, 96.9MB/s, 8.64MB/s  ]
Loading weights: 100% 223/223 [00:00<00:00, 4948.36it/s]
epoch 0 step 0/2 loss 0.8439
Writing model shards: 100% 1/1 [00:00<00:00,  7.63it/s]
{"epoch": 0, "train_loss": 0.7386539280414581, "val_iou": 0.47675304786919875, "best_iou": 0.47675304786919875, "seconds": 7.1, "device": "Tesla T4"}


In [17]:
!python -m minifsd.train.dino --train /content/data/v9/train --val /content/data/v9/val --run-dir {DRIVE}/runs/e2b-dinov2-s

Loading weights: 100% 223/223 [00:00<00:00, 4744.21it/s]
epoch 0 step 0/282 loss 0.8654
epoch 0 step 100/282 loss 0.0542
epoch 0 step 200/282 loss 0.0492
Writing model shards: 100% 1/1 [00:00<00:00,  3.73it/s]
{"epoch": 0, "train_loss": 0.07715453000863393, "val_iou": 0.9475390970630674, "best_iou": 0.9475390970630674, "seconds": 63.9, "device": "Tesla T4"}
epoch 1 step 0/282 loss 0.0500
epoch 1 step 100/282 loss 0.0451
epoch 1 step 200/282 loss 0.0390
Writing model shards: 100% 1/1 [00:00<00:00,  4.24it/s]
{"epoch": 1, "train_loss": 0.04695285337169965, "val_iou": 0.951632105349861, "best_iou": 0.951632105349861, "seconds": 59.9, "device": "Tesla T4"}
epoch 2 step 0/282 loss 0.0441
epoch 2 step 100/282 loss 0.0428
epoch 2 step 200/282 loss 0.0423
Writing model shards: 100% 1/1 [00:00<00:00,  4.57it/s]
{"epoch": 2, "train_loss": 0.04341928028729227, "val_iou": 0.9531209878166091, "best_iou": 0.9531209878166091, "seconds": 60.5, "device": "Tesla T4"}
epoch 3 step 0/282 loss 0.0356
epoch

In [18]:
!cat {DRIVE}/runs/e2b-dinov2-s/log.jsonl

{"epoch": 0, "train_loss": 0.07715453000863393, "val_iou": 0.9475390970630674, "best_iou": 0.9475390970630674, "seconds": 63.9, "device": "Tesla T4"}
{"epoch": 1, "train_loss": 0.04695285337169965, "val_iou": 0.951632105349861, "best_iou": 0.951632105349861, "seconds": 59.9, "device": "Tesla T4"}
{"epoch": 2, "train_loss": 0.04341928028729227, "val_iou": 0.9531209878166091, "best_iou": 0.9531209878166091, "seconds": 60.5, "device": "Tesla T4"}
{"epoch": 3, "train_loss": 0.04156620032257504, "val_iou": 0.954332267878117, "best_iou": 0.954332267878117, "seconds": 61.0, "device": "Tesla T4"}
{"epoch": 4, "train_loss": 0.04093136115868886, "val_iou": 0.9550105269777807, "best_iou": 0.9550105269777807, "seconds": 61.0, "device": "Tesla T4"}
{"epoch": 5, "train_loss": 0.039966086943944296, "val_iou": 0.9555099225158602, "best_iou": 0.9555099225158602, "seconds": 60.9, "device": "Tesla T4"}
{"epoch": 6, "train_loss": 0.039414391958051256, "val_iou": 0.9552632876562013, "best_iou": 0.955509922

## E2c: both floor models retrained on v10 (furniture)

Same recipes as E2a and E2b, new data. Run the settings and unpack cells
first so v10 is on the session disk. DINOv2 first: it is the quicker run.

In [ ]:
!python -m minifsd.train.dino --train /content/data/v10/train --val /content/data/v10/val --run-dir {DRIVE}/runs/e2c-dinov2-s-v10

In [ ]:
!python -m minifsd.train.segformer --train /content/data/v10/train --val /content/data/v10/val --run-dir {DRIVE}/runs/e2c-segformer-b0-v10

In [ ]:
!tail -n 3 {DRIVE}/runs/e2c-dinov2-s-v10/log.jsonl {DRIVE}/runs/e2c-segformer-b0-v10/log.jsonl